In [ ]:
# Run this cell if using Google Colab or if pyapprox is not installed locally
try:
    import pyapprox
except ImportError:
    !pip install "pyapprox[all]" -q
    # If the PyPI install fails, uncomment the line below to install from source:
    # !pip install "pyapprox[all] @ git+https://github.com/sandialabs/pyapprox.git" -q



## Learning Objectives

After completing this tutorial, you will be able to:

- Explain Sobol sensitivity indices as a decomposition of output variance into input contributions
- Distinguish first-order indices (direct effects) from total-order indices (effects including interactions)
- Interpret sensitivity results to guide resource allocation for uncertainty reduction


## Prerequisites

Complete [Forward UQ](forward_uq.ipynb) before this tutorial.


## Motivation: Which Parameters Matter?

Forward UQ tells us *how much* total uncertainty there is in a model's outputs. Now we ask a different question: **which uncertain inputs are responsible for that variability?**

If we can identify the dominant inputs, we can:

- **Focus resources** on reducing those uncertainties (better measurements, tighter manufacturing tolerances)
- **Fix unimportant inputs** at nominal values, simplifying the model
- **Understand** the model's input-output structure

**Sensitivity Analysis (SA)** provides a principled framework for answering this question.

In [ ]:
#| echo: false
import numpy as np
import matplotlib.pyplot as plt
from pyapprox.util.backends.numpy import NumpyBkd
from pyapprox.benchmarks import cantilever_beam_1d
from pyapprox.surrogates.affine.expansions.pce import (
    create_pce_from_marginals,
)
from pyapprox.surrogates.affine.expansions.fitters.least_squares import (
    LeastSquaresFitter,
)
from pyapprox.sensitivity import PolynomialChaosSensitivityAnalysis

bkd = NumpyBkd()

# Set up beam benchmark with 3 KLE terms
benchmark = cantilever_beam_1d(bkd, num_kle_terms=3)
model = benchmark.function()
prior = benchmark.prior()
marginals = prior.marginals()
nvars = model.nvars()
nqoi = model.nqoi()

qoi_labels = [
    r"$\delta_{\mathrm{tip}}$",
    r"$\sigma_{\mathrm{int}}$",
    r"$\kappa_{\max}$",
]
input_labels = [rf"$\xi_{{{ii+1}}}$" for ii in range(nvars)]

# Fit degree-4 PCE
N_train = 150
np.random.seed(42)
samples_train = prior.rvs(N_train)
values_train = model(samples_train)

degree = 4
pce = create_pce_from_marginals(marginals, degree, bkd, nqoi=nqoi)
ls_fitter = LeastSquaresFitter(bkd)
pce = ls_fitter.fit(pce, samples_train, values_train).surrogate()

# Compute sensitivity indices
sa = PolynomialChaosSensitivityAnalysis(pce)
main_effects = bkd.to_numpy(sa.main_effects())   # (nvars, nqoi)
total_effects = bkd.to_numpy(sa.total_effects())  # (nvars, nqoi)

## Variance Decomposition (ANOVA)

The key idea behind variance-based sensitivity analysis is **Analysis of Variance (ANOVA)**. The total output variance can be decomposed into contributions from individual inputs and their interactions:

$$
\mathbb{V}_{\boldsymbol{\theta}}[q] = \sum_{i} V_i + \sum_{i<j} V_{ij} + \cdots + V_{1,2,\ldots,d_\theta}
$$

where:

- $V_i = \mathbb{V}_{\theta_i}[\mathbb{E}_{\theta_{\sim i}}[q | \theta_i]]$ is the variance due to input $\theta_i$ alone
- $V_{ij}$ is the additional variance from the interaction between $\theta_i$ and $\theta_j$
- Higher-order terms capture three-way and beyond interactions

@fig-variance-decomposition shows how the variance of tip deflection decomposes across the three KLE inputs.

In [ ]:
#| echo: false
#| fig-cap: "Variance decomposition for beam tip deflection. Each slice shows the fraction of total variance attributable to each KLE input. The first KLE term dominates because it represents the lowest-frequency spatial mode of the stiffness field."
#| label: fig-variance-decomposition

from pyapprox.tutorial_figures._sensitivity import plot_variance_decomposition

qoi_plot = 0  # tip deflection
fig, ax = plt.subplots(figsize=(7, 4.5))
plot_variance_decomposition(main_effects, qoi_plot, input_labels,
                            qoi_labels, ax)
plt.tight_layout()
plt.show()

## First-Order and Total-Order Indices

The ANOVA decomposition motivates two key measures of input importance.

### First-Order Sobol Index

The **first-order Sobol index** $S_{i}$ measures the fraction of output variance explained by input $\theta_i$ acting alone:

$$
S_{i} = \frac{V_i}{\mathbb{V}_{\boldsymbol{\theta}}[q]} = \frac{\mathbb{V}_{\theta_i}[\mathbb{E}_{\theta_{\sim i}}[q | \theta_i]]}{\mathbb{V}_{\boldsymbol{\theta}}[q]}
$$

- $S_{i} \approx 1$: input $i$ explains nearly all variance by itself
- $S_{i} \approx 0$: input $i$ has little direct effect on variance

### Total-Order Sobol Index

The **total-order Sobol index** $S_{i}^T$ captures the total contribution of input $\theta_i$, including all interactions with other inputs:

$$
S_{i}^T = 1 - \frac{\mathbb{V}_{\theta_{\sim i}}[\mathbb{E}_{\theta_i}[q | \theta_{\sim i}]]}{\mathbb{V}_{\boldsymbol{\theta}}[q]}
$$

where $\theta_{\sim i}$ denotes all inputs except $\theta_i$.

Key properties:

- $S_{i}^T \geq S_{i}$ always
- If $S_{i}^T \approx S_{i}$: input $i$ has few interactions with other inputs
- If $S_{i}^T \gg S_{i}$: input $i$ participates in strong interactions
- $\sum_i S_{i} \leq 1$ (equality iff no interactions)
- $\sum_i S_{i}^T \geq 1$ (equality iff no interactions)

@fig-sobol-bar-chart shows both indices for the beam QoIs that have nonzero variance: tip deflection and maximum curvature.

In [ ]:
#| echo: false
#| fig-cap: "First-order and total-order Sobol indices for tip deflection and maximum curvature. When the bars are similar in height, interactions are weak. When the total-order bar is much taller, the input participates in interactions with other inputs."
#| label: fig-sobol-bar-chart

from pyapprox.tutorial_figures._sensitivity import plot_sobol_bar_chart

# Plot only QoIs with nonzero variance (skip sigma_int, index 1)
plot_qois = [0, 2]

fig, axes = plt.subplots(1, len(plot_qois), figsize=(10, 4), sharey=True)
plot_sobol_bar_chart(main_effects, total_effects, plot_qois,
                     input_labels, qoi_labels, nvars, axes)
plt.tight_layout()
plt.show()

::: {.callout-note}
## Why is $\sigma_{\mathrm{int}}$ missing?
The integrated bending stress $\sigma_{\mathrm{int}} = \int_0^L \sigma(x)\, dx$ has **zero variance** --- it is constant regardless of the KLE inputs. This is because the internal bending moment distribution $M(x)$ is determined entirely by the applied load and boundary conditions (static equilibrium), not by the stiffness field $EI(x)$. The stiffness affects *how much the beam deflects* and *how curvature distributes*, but the total integrated stress $\int M(x)\, y/I(x) \cdot dx$ simplifies to a load-dependent constant for this model. A QoI with zero variance has undefined Sobol indices, so it is omitted from the sensitivity analysis.
:::


## Visual Confirmation

A quick visual check supports the sensitivity indices. @fig-scatter-dominant shows scatter plots of the QoI against the most influential and least influential KLE inputs. The dominant input shows a clear functional trend, while the non-influential input shows no pattern.

In [ ]:
#| echo: false
#| fig-cap: "Scatter plots confirming sensitivity rankings. Left: QoI vs. dominant KLE input (clear trend explains high S_i). Right: QoI vs. least influential input (no trend, low S_i)."
#| label: fig-scatter-dominant

from pyapprox.tutorial_figures._sensitivity import plot_scatter_dominant

qoi_plot = 0
np.random.seed(99)
N_scatter = 500
scatter_samples = prior.rvs(N_scatter)
scatter_values = bkd.to_numpy(model(scatter_samples))
scatter_samples_np = bkd.to_numpy(scatter_samples)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
plot_scatter_dominant(scatter_samples_np, scatter_values, main_effects,
                      qoi_plot, input_labels, qoi_labels, ax1, ax2)
plt.tight_layout()
plt.show()

## Key Takeaways

- **Variance decomposition** splits output variance into contributions from individual inputs and their interactions
- **First-order** $S_{i}$ measures the direct (additive) effect of input $i$
- **Total-order** $S_{i}^T$ measures the total effect including all interactions --- use this for deciding which inputs to fix
- $\sum_i S_{i} < 1$ indicates the presence of interactions between inputs
- A QoI with **zero variance** (e.g., determined by static equilibrium) has undefined Sobol indices


## Exercises

1. **(Easy)** From @fig-sobol-bar-chart, which KLE input is most important for each QoI? Do the rankings differ between tip deflection and maximum curvature?

2. **(Medium)** For a given QoI, what does it mean if $S_{i}^T \gg S_{i}$? Use @fig-sobol-bar-chart to identify whether any input exhibits strong interactions.

3. **(Challenge)** The sum $\sum_i S_{i}$ measures how additive the model is. Compute this sum for both QoIs. Which has stronger interactions? What physical mechanism might explain this?


## Next Steps

Continue with:

- [Sobol Sensitivity Analysis](sobol_sensitivity_analysis.ipynb) --- Monte Carlo estimation of Sobol indices (no surrogate needed)
- [PCE-Based Sensitivity](pce_sensitivity.ipynb) --- Detailed API for PCE sensitivity including interaction terms
- [Morris Screening](morris_screening.ipynb) --- Cheap screening to identify important inputs before Sobol analysis